# Lecture 7 Activity – Does where a voter lives catch what the surname misses?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

In lecture, the largest group for a voter's surname guessed the recorded race
right for 79.1 percent of Florida voters. For Black voters, it was right for
only 16.2 percent. Many Black Americans hold surnames that more white Americans hold,
such as WILLIAMS and JOHNSON.

The voter file also records where each voter lives: `county` and `VTD`, the
voting district. A Census table gives, for each district, the share of its
residents in each group.

> **Does a guess from the voter's district catch the Black voters that the
> surname guess misses, and what does it get wrong instead?**

### Reminders

- `voters`: one row per voter. `districts`: one row per voting district, with
  the shares `white`, `black`, `hispanic`, `api`, and `others` (from 0 to 1).
- A `VTD` number repeats across counties, so only the pair (`county`, `VTD`)
  identifies a district.
- Compare the two guesses on the **same voters**.
- In the starter cells, replace each `...` with code.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 8)

# Sharp figures on high-resolution screens, from lecture 6.
%config InlineBackend.figure_format = 'retina'

voters = pd.read_csv('../../data/raw/FLVoters.csv', keep_default_na=False, na_values=['NA'])
names = pd.read_csv('../../data/raw/names.csv', keep_default_na=False)
districts = pd.read_csv('../../data/raw/FLCensusVTD.csv')

# The lecture's surname guess: the largest group for each surname.
pct_cols = ['pctwhite', 'pctblack', 'pctapi', 'pcthispanic', 'pctothers']
to_race = {'pctwhite': 'white', 'pctblack': 'black', 'pctapi': 'asian',
           'pcthispanic': 'hispanic', 'pctothers': 'other'}
names = names.assign(name_guess=names[pct_cols].idxmax(axis=1).map(to_race))

voters = voters.merge(names[['surname', 'name_guess']], on='surname',
                      how='left', validate='many_to_one')

# Check: 10,000 voters, and 8,806 of them have a surname guess.
len(voters), voters['name_guess'].notna().sum()

## Task 1 · Join each voter to their district

First check that the pair (`county`, `VTD`) appears only once in
`districts`. Then merge `districts` onto `voters` with a left join on **both**
columns, with `validate` and `indicator=True`, as `with_district`.

How many rows does `with_district` have, and how many voters found their
district?



<details><summary>Hint</summary>

A two-column key is a list: `on=['county', 'VTD']`. Many voters live in one
district, so the relationship is `'many_to_one'`.

</details>

In [ ]:
print('repeated (county, VTD) pairs:',
      districts.duplicated(subset=[..., ...]).sum())

with_district = voters.merge(districts, on=..., how='left',
                             validate=..., indicator=True)
print('rows:', len(with_district))
with_district['_merge'].value_counts()

*Your answer:*

## Task 2 · A guess from the district

Guess each voter's race as the **largest group in their district**. Use
`idxmax(axis=1)` on the five share columns and rename the groups with `.map`,
so `api` becomes `asian` and `others` becomes `other`.

Then keep the voters with **both** a recorded race and a surname guess, as
`known`, and compute the share for whom the district guess is right. How does
it compare with the surname guess on the same voters?



<details><summary>Hint</summary>

The lecture's pattern: `df[cols].idxmax(axis=1)` gives a column name per row,
and `.map(dict)` renames it. `known` needs two conditions: `race` is not
missing, and `name_guess` is not missing (`.notna()`).

</details>

In [ ]:
district_cols = ['white', 'black', 'hispanic', 'api', 'others']
to_voter = {'white': 'white', 'black': 'black', 'hispanic': 'hispanic',
            'api': 'asian', 'others': 'other'}

with_district = with_district.assign(
    district_guess=with_district[district_cols].idxmax(axis=1).map(...))

known = with_district.loc[with_district['race'].notna()
                          & ...]
print('voters:', len(known))
print('surname guess right: ', round((known['name_guess'] == known['race']).mean(), 3))
print('district guess right:', ...)

*Your answer:*

## Task 3 · ❓ Group by group

In lecture, the overall accuracy hid the
difference between groups. Build one table with one row
per recorded race and three named columns: `voters` (how many), `surname`
(share the surname guess gets right), and `district` (share the district
guess gets right). Then draw `surname` and `district` side by side as a
horizontal bar chart.

For which groups does the district do better, and for which does it do
worse?



<details><summary>Hint</summary>

Add two `True`/`False` columns with `.assign`, then use named aggregation:
`.groupby('race').agg(voters=('race', 'size'), surname=('name_right',
'mean'), ...)`. `table[['surname', 'district']].plot.barh()` draws both
columns.

</details>

In [ ]:
by_group = (
    known
    .assign(name_right=known['name_guess'] == known['race'],
            district_right=...)
    .groupby('race')
    .agg(voters=('race', 'size'),
         surname=('name_right', 'mean'),
         district=...)
)
by_group.round(3)

*Your answer:*

## Task 4 · What the guesses cannot tell you

Write two or three sentences. When can a district guess be right for a Black
voter, and when must it be wrong? What can neither guess establish about one
voter, and what does that mean for counting voters by race?

*Your answer:*

## If you finish early · Combine the two

A simple version of Bayesian Improved Surname Geocoding (BISG)
multiplies the two pieces of information. For each group,
compute a score:

> surname share × district share ÷ the group's national share

and guess the group with the largest score. Dividing by the national share
keeps a large group from winning everywhere. Compute the accuracy by race and
compare it with Task 3.



<details><summary>Hint</summary>

Merge the five surname shares onto `known` first. The national share of each
group is its share of all people in `names`, weighted by `count`; the starter
computes it for you.

</details>

In [ ]:
# National share of each group, over all people in the surname list.
national = (names[pct_cols].mul(names['count'], axis=0).sum()
            / names['count'].sum())

both = known.merge(names[['surname'] + pct_cols], on='surname',
                   how='left', validate='many_to_one')
pairs = {'white': ('pctwhite', 'white'), 'black': ('pctblack', 'black'),
         'hispanic': ('pcthispanic', 'hispanic'), 'asian': ('pctapi', 'api'),
         'other': ('pctothers', 'others')}
scores = pd.DataFrame({race: ... for race, (p, d) in pairs.items()})
both = both.assign(combined_guess=scores.idxmax(axis=1))
...

*Your answer:*